In [1]:
import pandas as pd

In [7]:
Messy_data=pd.read_csv('messy_data.csv')
Messy_data.head(20)

,patient_id,first_name,last_name,gender,age,condition,admission_date,phone,weight,is_smoker,treatment_cost,email
0,PT900295,HIROSHI,Williams,Female,51.0,migraine,04-28-2023,7257273491,105.9 kg,N,3960.83,hiroshi.williams295@example.com
1,PT900075,Sofia,Nguyen,Male,83,ASTHMA,24-May-24,531.879.2068,105.3,N,$510.81,sofia.nguyen75@example.com
2,PT900177,Carlos,WILLIAMS,M,16,DIABETES,Jul 13 2023,438-180-8444,65.4 kg,1,875.51,carlos.williams177@example.com
3,PT900030,Elizabeth,Davis,F,92.0,asthma,2024-04-11,530-442-7454,60.5,1,"3,317.58",elizabeth.davis30@example.com
4,PT900360,Sofia,Brown,Female,56 yrs,Hypertension,2024-08-05,8964449377,68.7 kg,No,1397.13,sofia.brown360@example.com
5,PT900272,Wei,Jones,M,3,Diabetis,2023-09-10,2872479217,80.0,N,"$1,880.02",wei.jones272@example.com
6,PT900155,Elizabeth,Brown,F,41.0,HYPERTENSION,Sep 13 2024,(386) 585-4192,101.4 kg,False,1290.19,elizabeth.brown155@example.com
7,PT900152,Amara,MILLER,FEMALE,41.0,ARTHRITIS,Aug 08 2023,3835219110,108.8,TRUE,"$1,398.36",amara.miller152@example.com
8,PT900165,David,Smith,Female,43 years old,MIGRAINE,07-19-2023,8732839356,0,N,"1,538.11",david.smith165@example.com
9,PT900175,Wei,Patel,Male,38,asthma,05-20-2023,591.905.8256,107.5,FALSE,"4,944.31",wei.patel175@example.com


#**This data has following issues --- Manual checking

    #**1--Inconsistent categorical casing/spacing — gender has Male
    #MALE, male, M, " Male", Female, FEMALE, female, F, Other, O
    
    #**2--Typos & inconsistent spelling, condition has Diabetes, 
    #diabetees, DIABETES, Diabetis, Hypertension, Hypertention, 
    #Arthritis, Arthritus, Migraine, Migrain, Asma

    #**3--Inconsistent date formats — admission_date mixes 2024-01-05, 
    #01/05/2024, Jan 05 2024, 05-Jan-24, 01-05-2024

    #**4--Inconsistent phone formats — (555) 123-4567, 555-123-4567, 
    #5551234567, 555.123.4567

    #**5--Inconsistent boolean representations — is_smoker has Yes/No
    #/Y/N/1/0/True/False in various cases

    #**6--Inconsistent name casing — some last_name values are ALL CAPS, 
    #some emails are uppercase

    #**7--Multiple missing-value representations — NaN, '', 'N/A', 'n/a',        'Unknown', 'unknown', 'null', 'NULL', '--', '?' all mean "missing" but        pandas won't recognize most of them automatically

    #**8--Mixed-type numeric column — age is stored as text: "45", "45 yrs",     "45.0", "45 years old"

    #**9--Impossible/outlier values — negative ages, age of 300, negative or     zero weight, weight of 500

    #**10--Currency stored as text — treatment_cost mixes "$1,200.50",           "1200.50", and plain numbers — can't do math on it as-is

    #**11--Extra whitespace padding — leading/trailing spaces in first_name,     patient_id, condition

    #**12--Duplicate rows — exact duplicates were inserted Near-duplicate        rows — same person, but with different casing/spacing making them look       different (a classic dedup challenge)



#*Programmatic cleaning

#**1--Inconsistent categorical casing/spacing — gender has Male, MALE, male, M, " Male", Female, FEMALE, female, F, Other, O

In [9]:
gender_map={'M': 'Male', ' Male': 'Male', 'MALE':'Male', 'male':'Male', 'FEMALE':'Female', 
            ' Female':'Female', 'female':'Female', 'F':'Female', 'O':'Other', 'Other':'Other'}

# Normalizing the dict keys the same way you normalize the data
gender_map_clean = {k.strip().lower(): v for k, v in gender_map.items()}

Messy_data['gender']=Messy_data['gender'].astype(str).str.strip().str.lower().map(gender_map_clean)
Messy_data.head(20)

,patient_id,first_name,last_name,gender,age,condition,admission_date,phone,weight,is_smoker,treatment_cost,email
0,PT900295,HIROSHI,Williams,Female,51.0,migraine,04-28-2023,7257273491,105.9 kg,N,3960.83,hiroshi.williams295@example.com
1,PT900075,Sofia,Nguyen,Male,83,ASTHMA,24-May-24,531.879.2068,105.3,N,$510.81,sofia.nguyen75@example.com
2,PT900177,Carlos,WILLIAMS,Male,16,DIABETES,Jul 13 2023,438-180-8444,65.4 kg,1,875.51,carlos.williams177@example.com
3,PT900030,Elizabeth,Davis,Female,92.0,asthma,2024-04-11,530-442-7454,60.5,1,"3,317.58",elizabeth.davis30@example.com
4,PT900360,Sofia,Brown,Female,56 yrs,Hypertension,2024-08-05,8964449377,68.7 kg,No,1397.13,sofia.brown360@example.com
5,PT900272,Wei,Jones,Male,3,Diabetis,2023-09-10,2872479217,80.0,N,"$1,880.02",wei.jones272@example.com
6,PT900155,Elizabeth,Brown,Female,41.0,HYPERTENSION,Sep 13 2024,(386) 585-4192,101.4 kg,False,1290.19,elizabeth.brown155@example.com
7,PT900152,Amara,MILLER,Female,41.0,ARTHRITIS,Aug 08 2023,3835219110,108.8,TRUE,"$1,398.36",amara.miller152@example.com
8,PT900165,David,Smith,Female,43 years old,MIGRAINE,07-19-2023,8732839356,0,N,"1,538.11",david.smith165@example.com
9,PT900175,Wei,Patel,Male,38,asthma,05-20-2023,591.905.8256,107.5,FALSE,"4,944.31",wei.patel175@example.com


#**2--Typos & inconsistent spelling, condition has Diabetes, diabetees,            DIABETES, Diabetis, Hypertension, Hypertention, Arthritis, Arthritus,        Migraine, Migrain, Asma

In [11]:
Messy_data['condition'].unique()

<StringArray>
[     'migraine',        'ASTHMA',      'DIABETES',        'asthma',
 ' Hypertension',      'Diabetis',  'HYPERTENSION',     'ARTHRITIS',
      'MIGRAINE',     'Arthritis',  'hypertension',      'Migraine',
   'Arthritis  ',       'Asthma ',       'Migrain',  'Hypertension',
        'Asthma',      'diabetes',     'Arthritus',             nan,
     'diabetees',          'Asma',   '  Diabetes ',  'Hypertention',
     'arthritis',      'Diabetes',       'Unknown',    ' Migraine ',
            '--',       'unknown',             '?']
Length: 31, dtype: str

In [13]:
condition_map={'migraine':'Migraine', 'ASTHMA':'Asthma', 'DIABETES':'Diabetes', 'asthma':'Asthma',
 ' Hypertension':'Hypertension', 'Diabetis':'Diabetes', 'HYPERTENSION':'Hypertension', 'ARTHRITIS':'Arthritis',
      'MIGRAINE':'Migraine', 'hypertension':'Hypertension', 'Arthritis  ':'Arthritis', 'Asthma ':'Asthma', 
        'Migrain':'Migraine', 'diabetes':'Diabetes', 'Arthritus':'Arthritis', 'nan':'NaN',
     'diabetees':'Diabetes', 'Asma':'Asthma', '  Diabetes ':'Diabetes', 'Hypertention':'Hypertension',
     'arthritis':'Arthritis', 'Migraine ':'Migraine'}

Messy_data['condition']=Messy_data['condition'].astype(str).str.strip().str.lower().map(condition_map)
Messy_data.head(20)

,patient_id,first_name,last_name,gender,age,condition,admission_date,phone,weight,is_smoker,treatment_cost,email
0,PT900295,HIROSHI,Williams,Female,51.0,Migraine,04-28-2023,7257273491,105.9 kg,N,3960.83,hiroshi.williams295@example.com
1,PT900075,Sofia,Nguyen,Male,83,Asthma,24-May-24,531.879.2068,105.3,N,$510.81,sofia.nguyen75@example.com
2,PT900177,Carlos,WILLIAMS,Male,16,Diabetes,Jul 13 2023,438-180-8444,65.4 kg,1,875.51,carlos.williams177@example.com
3,PT900030,Elizabeth,Davis,Female,92.0,Asthma,2024-04-11,530-442-7454,60.5,1,"3,317.58",elizabeth.davis30@example.com
4,PT900360,Sofia,Brown,Female,56 yrs,Hypertension,2024-08-05,8964449377,68.7 kg,No,1397.13,sofia.brown360@example.com
5,PT900272,Wei,Jones,Male,3,NaN,2023-09-10,2872479217,80.0,N,"$1,880.02",wei.jones272@example.com
6,PT900155,Elizabeth,Brown,Female,41.0,Hypertension,Sep 13 2024,(386) 585-4192,101.4 kg,False,1290.19,elizabeth.brown155@example.com
7,PT900152,Amara,MILLER,Female,41.0,Arthritis,Aug 08 2023,3835219110,108.8,TRUE,"$1,398.36",amara.miller152@example.com
8,PT900165,David,Smith,Female,43 years old,Migraine,07-19-2023,8732839356,0,N,"1,538.11",david.smith165@example.com
9,PT900175,Wei,Patel,Male,38,Asthma,05-20-2023,591.905.8256,107.5,FALSE,"4,944.31",wei.patel175@example.com


In [14]:
Messy_data['condition'].unique()

<StringArray>
['Migraine', 'Asthma', 'Diabetes', 'Hypertension', nan, 'Arthritis']
Length: 6, dtype: str

#**3--Inconsistent date formats — admission_date mixes 2024-01-05, 
#01/05/2024, Jan 05 2024, 05-Jan-24, 01-05-2024

In [15]:
Messy_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 429 entries, 0 to 428
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   patient_id      425 non-null    str  
 1   first_name      425 non-null    str  
 2   last_name       425 non-null    str  
 3   gender          388 non-null    str  
 4   age             410 non-null    str  
 5   condition       328 non-null    str  
 6   admission_date  425 non-null    str  
 7   phone           414 non-null    str  
 8   weight          416 non-null    str  
 9   is_smoker       414 non-null    str  
 10  treatment_cost  409 non-null    str  
 11  email           420 non-null    str  
dtypes: str(12)
memory usage: 40.3 KB


In [19]:
Messy_data['admission_date']=pd.to_datetime(Messy_data['admission_date'], format='mixed', errors='coerce')

In [20]:
Messy_data.head(4)

,patient_id,first_name,last_name,gender,age,condition,admission_date,phone,weight,is_smoker,treatment_cost,email
0,PT900295,HIROSHI,Williams,Female,51.0,Migraine,2023-04-28,7257273491,105.9 kg,N,3960.83,hiroshi.williams295@example.com
1,PT900075,Sofia,Nguyen,Male,83,Asthma,2024-05-24,531.879.2068,105.3,N,$510.81,sofia.nguyen75@example.com
2,PT900177,Carlos,WILLIAMS,Male,16,Diabetes,2023-07-13,438-180-8444,65.4 kg,1,875.51,carlos.williams177@example.com
3,PT900030,Elizabeth,Davis,Female,92.0,Asthma,2024-04-11,530-442-7454,60.5,1,"3,317.58",elizabeth.davis30@example.com


In [21]:
Messy_data['admission_date'].unique()

<DatetimeArray>
['2023-04-28 00:00:00', '2024-05-24 00:00:00', '2023-07-13 00:00:00',
 '2024-04-11 00:00:00', '2024-08-05 00:00:00', '2023-09-10 00:00:00',
 '2024-09-13 00:00:00', '2023-08-08 00:00:00', '2023-07-19 00:00:00',
 '2023-05-20 00:00:00',
 ...
 '2024-06-30 00:00:00', '2025-04-07 00:00:00', '2024-09-24 00:00:00',
 '2025-04-26 00:00:00', '2025-01-08 00:00:00', '2023-01-16 00:00:00',
 '2024-06-03 00:00:00', '2023-08-02 00:00:00', '2024-07-20 00:00:00',
 '2023-11-05 00:00:00']
Length: 322, dtype: datetime64[us]

In [23]:
print(Messy_data['admission_date'].isna().sum())

4


#**5--Inconsistent boolean representations — is_smoker has Yes/No
#/Y/N/1/0/True/False in various cases

In [24]:
Messy_data['is_smoker'].unique()

<StringArray>
[      'N',       '1',      'No',   'False',    'TRUE',   'FALSE',      'no',
     'YES',     'Yes',    'True',    'true',      'NO',       'Y',      '--',
 'unknown',       nan,   'false',     'yes',       '0',       '?', 'Unknown']
Length: 21, dtype: str

In [30]:
smoking_map={'N':'No', '1':'Yes', 'No':'No', 'False':'No', 'TRUE':'Yes', 'FALSE':'No', 'no':'No',
     'YES':'Yes', 'nan':'NaN', 'Yes':'Yes', 'True':'Yes', 'true':'Yes', 'NO':'No', 'Y':'Yes', 'false':'No', 'yes':'Yes', '0':'No'}

smoking_clean={k.strip().lower():v for k, v in smoking_map.items()}

Messy_data['is_smoker']=Messy_data['is_smoker'].astype(str).str.strip().str.lower().map(smoking_clean)
Messy_data.head(6)

,patient_id,first_name,last_name,gender,age,condition,admission_date,phone,weight,is_smoker,treatment_cost,email
0,PT900295,HIROSHI,Williams,Female,51.0,Migraine,2023-04-28,7257273491,105.9 kg,NaN,3960.83,hiroshi.williams295@example.com
1,PT900075,Sofia,Nguyen,Male,83,Asthma,2024-05-24,531.879.2068,105.3,NaN,$510.81,sofia.nguyen75@example.com
2,PT900177,Carlos,WILLIAMS,Male,16,Diabetes,2023-07-13,438-180-8444,65.4 kg,Yes,875.51,carlos.williams177@example.com
3,PT900030,Elizabeth,Davis,Female,92.0,Asthma,2024-04-11,530-442-7454,60.5,Yes,"3,317.58",elizabeth.davis30@example.com
4,PT900360,Sofia,Brown,Female,56 yrs,Hypertension,2024-08-05,8964449377,68.7 kg,No,1397.13,sofia.brown360@example.com
5,PT900272,Wei,Jones,Male,3,NaN,2023-09-10,2872479217,80.0,NaN,"$1,880.02",wei.jones272@example.com
